# 1. Data pipeline

##### Running this pipeline on copy_of_sample data will produce feature engineered data that can be used to upload to the app and generate predictions

In [24]:
import pandas as pd

# If you choose training, it will fetch the training data and create a feature engineered tarining data for model training.
# If you choose production, it will fetch a sample production data, feature engineer it similarly and the resultant data can be used as an input for the app
x='training' # another option: production.

if x=='training':
    output_path='./data/processed/customer_features.csv'
    raw_data_path='./data/raw/training.csv'
elif x=='production'
    output_path='./data/processed/customer_features_prod_data.csv'
    raw_data_path='./data/raw/copy_of_sample.csv'

end_date = pd.Timestamp('2021-10-29', tz='UTC')

## Read the data and change data types

In [25]:
transactions=pd.read_csv(raw_data_path)
transactions = transactions.rename(columns={'product_type': 'product_code'})
transactions = transactions.drop_duplicates().copy()

for column in ['acquired_date', 'transaction_time']:
    transactions[column] = pd.to_datetime(
        transactions[column].str.replace(' UTC', '', regex=False),
        format='mixed', errors='coerce', utc=True)
transactions['value'] = pd.to_numeric(transactions['value'], errors='coerce')

transactions = transactions.sort_values('transaction_time', kind='stable')

## Create customer-level features

In [26]:
customers = transactions.drop_duplicates('customer_id').set_index('customer_id').copy()
customers = customers.rename(columns={'transaction_time': 'first_purchase_time', 'value': 'first_purchase_amount'})

extract_start = transactions['transaction_time'].min().normalize()
new_customer = (customers['acquired_date'] >= extract_start) & (customers['acquired_date'] <= customers['first_purchase_time'])
customers = customers.loc[new_customer].copy()

customers['days_to_first_purchase'] = (
    customers['first_purchase_time'] - customers['acquired_date']).dt.total_seconds() / 86400
customers['purchase_hour'] = customers['first_purchase_time'].dt.hour
customers['purchase_weekday'] = customers['first_purchase_time'].dt.day_name()
customers['acquisition_hour'] = customers['acquired_date'].dt.hour
customers['acquisition_weekday'] = customers['acquired_date'].dt.day_name()

categorical_columns = ['parent_region_id', 'child_region_id', 'product_code',
                       'acquired_by', 'purchase_weekday', 'acquisition_weekday']
for column in categorical_columns:
    customers[column] = customers[column].fillna('Unknown').astype('category')

customers.head()

,acquired_by,acquired_date,first_purchase_time,product_code,parent_region_id,first_purchase_amount,child_region_id,days_to_first_purchase,purchase_hour,purchase_weekday,acquisition_hour,acquisition_weekday
customer_id,,,,,,,,,,,,
59e72d7b09073db4b5b8fe0b3ffd6a34,OFFLINE,2021-01-27 11:09:13.391000+00:00,2021-02-10 02:30:00+00:00,78,1EF,83,740CB1CA2BD83B3276B092195A6EB883,13.639428,2,Wednesday,11,Wednesday
0acd250a63e4c252d301a87d954defb0,OFFLINE,2021-02-08 11:25:32.346000+00:00,2021-02-10 08:13:00+00:00,78,1EF,60,08A9B54E00545ECCD369B8E5D2692E9F,1.866292,8,Wednesday,11,Monday
1b266bec0be91997a9b7ca9ca1a1a143,OFFLINE,2021-01-16 08:33:03.642000+00:00,2021-02-10 10:08:00+00:00,78,300,96,C0DD698BE0C1EA71FA3045950BEEC64B,25.065930,10,Wednesday,8,Saturday
fea369eafbabdae8e82be476ca2bb117,OFFLINE,2021-01-20 17:21:13.153000+00:00,2021-02-10 10:31:00+00:00,88,07B,80,EE735FFED3A6072D13A30636E658A03A,20.715126,10,Wednesday,17,Wednesday
6b01156e909e9a0b157c08554cd6a35e,ONLINE,2021-01-31 05:55:03.827000+00:00,2021-02-10 10:40:00+00:00,78,07B,138,294B021232C509826A57AC3136B8ABDD,10.197872,10,Wednesday,5,Sunday


## Calculate the binary target
Any new customer with 3 distinct dates purchases within the next 7 days of their first purchase is marked 1 as the target/sticky label

In [27]:
customers['first_purchase_date'] = customers['first_purchase_time'].dt.normalize()
customers['followup_end'] = customers['first_purchase_date'] + pd.Timedelta(days=7)
customers['label_available'] = customers['followup_end'] <= end_date

future = transactions[['customer_id', 'transaction_time']].join(
    customers[['first_purchase_date', 'followup_end']], on='customer_id')
future['purchase_date'] = future['transaction_time'].dt.normalize()
future = future.loc[(future['purchase_date'] > future['first_purchase_date']) &
                    (future['purchase_date'] <= future['followup_end'])]
return_days = future.groupby('customer_id')['purchase_date'].nunique()

customers['future_return_days'] = customers.index.to_series().map(return_days).fillna(0)
customers['future_return_days'] = customers['future_return_days'].where(customers['label_available']).astype('Int64')
customers['sticky_target'] = (customers['future_return_days'] >= 3).astype('Int64')

customers.head()

,acquired_by,acquired_date,first_purchase_time,product_code,parent_region_id,first_purchase_amount,child_region_id,days_to_first_purchase,purchase_hour,purchase_weekday,acquisition_hour,acquisition_weekday,first_purchase_date,followup_end,label_available,future_return_days,sticky_target
customer_id,,,,,,,,,,,,,,,,,
59e72d7b09073db4b5b8fe0b3ffd6a34,OFFLINE,2021-01-27 11:09:13.391000+00:00,2021-02-10 02:30:00+00:00,78,1EF,83,740CB1CA2BD83B3276B092195A6EB883,13.639428,2,Wednesday,11,Wednesday,2021-02-10 00:00:00+00:00,2021-02-17 00:00:00+00:00,True,0,0
0acd250a63e4c252d301a87d954defb0,OFFLINE,2021-02-08 11:25:32.346000+00:00,2021-02-10 08:13:00+00:00,78,1EF,60,08A9B54E00545ECCD369B8E5D2692E9F,1.866292,8,Wednesday,11,Monday,2021-02-10 00:00:00+00:00,2021-02-17 00:00:00+00:00,True,0,0
1b266bec0be91997a9b7ca9ca1a1a143,OFFLINE,2021-01-16 08:33:03.642000+00:00,2021-02-10 10:08:00+00:00,78,300,96,C0DD698BE0C1EA71FA3045950BEEC64B,25.065930,10,Wednesday,8,Saturday,2021-02-10 00:00:00+00:00,2021-02-17 00:00:00+00:00,True,0,0
fea369eafbabdae8e82be476ca2bb117,OFFLINE,2021-01-20 17:21:13.153000+00:00,2021-02-10 10:31:00+00:00,88,07B,80,EE735FFED3A6072D13A30636E658A03A,20.715126,10,Wednesday,17,Wednesday,2021-02-10 00:00:00+00:00,2021-02-17 00:00:00+00:00,True,0,0
6b01156e909e9a0b157c08554cd6a35e,ONLINE,2021-01-31 05:55:03.827000+00:00,2021-02-10 10:40:00+00:00,78,07B,138,294B021232C509826A57AC3136B8ABDD,10.197872,10,Wednesday,5,Sunday,2021-02-10 00:00:00+00:00,2021-02-17 00:00:00+00:00,True,0,0


## Save the customer dataset

Features and target columns are ready and now we can save the file

In [28]:
customers.to_csv(output_path, index=True)
print(f'Saved {len(customers):,} customer rows to {output_path}')
print(f"{customers['sticky_target'].notna().sum():,} customers have complete labels for modeling")

Saved 1,766 customer rows to ./data/processed/customer_features_prod_data.csv
1,022 customers have complete labels for modeling
